# DataFrame & Column
##### 목표
1. 열 생성
1. 열 부분 집합 생성
1. 열 추가 또는 교체
1. 행 부분 집합 생성
1. 행 정렬

##### 방법
- <a href="https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/dataframe.html" target="_blank">DataFrame</a>: **`select`**, **`selectExpr`**, **`drop`**, **`withColumn`**, **`withColumnRenamed`**, **`filter`**, **`distinct`**, **`limit`**, **`sort`**
- <a href="https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/column.html" target="_blank">Column</a>: **`alias`**, **`isin`**, **`cast`**, **`isNotNull`**, **`desc`**, **`operators`**

In [0]:
%run ../Includes/Classroom-Setup-00.03

Python interpreter will be restarted.
Python interpreter will be restarted.


Resetting the learning environment:
| No action taken

Skipping install of existing datasets to "dbfs:/mnt/dbacademy-datasets/data-engineering-with-databricks/v02"

Validating the locally installed datasets:
| listing local files...(3 seconds)
| validation completed...(3 seconds total)

Creating & using the schema "3dt005_nuk5_da_dewd" in the catalog "hive_metastore"...(1 seconds)

Cloning the sales table from dbfs:/mnt/dbacademy-datasets/data-engineering-with-databricks/v02/ecommerce/delta/sales_hist...(5 seconds / 10,510 records)
Cloning the events table from dbfs:/mnt/dbacademy-datasets/data-engineering-with-databricks/v02/ecommerce/delta/events_hist...(4 seconds / 485,696 records)
Cloning the events_raw table from dbfs:/mnt/dbacademy-datasets/data-engineering-with-databricks/v02/ecommerce/delta/events_raw...(5 seconds / 2,252 records)
Cloning the users table from dbfs:/mnt/dbacademy-datasets/data-engineering-with-databricks/v02/ecommerce/delta/users_hist...(4 seconds / 251,501 reco


events 데이터 세트를 사용해 보겠습니다.

In [0]:
events_df = spark.table("events")
display(events_df.limit(10))

device,ecommerce,event_name,event_previous_timestamp,event_timestamp,geo,items,traffic_source,user_first_touch_timestamp,user_id
Android,"List(null, null, null)",mattresses,1593614063954129,1593614089359899,"List(Attleboro, MA)",List(),google,1593614037088511,UA000000106525232
Android,"List(null, null, null)",main,null,1593613901473050,"List(Rockport, TX)",List(),google,1593613901473050,UA000000106524533
Android,"List(null, null, null)",add_item,1593595595984555,1593595620017592,"List(De Kalb, TX)","List(List(null, M_STAN_Q, Standard Queen Mattress, 1045.0, 1045.0, 1))",instagram,1593595391984879,UA000000106466918
Linux,"List(null, null, null)",main,null,1593611755190083,"List(Warwick, RI)",List(),google,1593611755190083,UA000000106513930
Linux,"List(null, null, null)",mattresses,null,1593613449856735,"List(Coeur d'Alene, ID)",List(),youtube,1593613449856735,UA000000106522140
macOS,"List(null, null, null)",guest,1593617658665328,1593617822519726,"List(Clovis, CA)","List(List(null, M_STAN_T, Standard Twin Mattress, 595.0, 595.0, 1))",facebook,1593616922295696,UA000000106541016
macOS,"List(null, null, null)",email_coupon,1593617459515520,1593618432116637,"List(St. Augustine, FL)",List(),google,1593616968692210,UA000000106541282
iOS,"List(null, null, null)",main,null,1593615063697854,"List(Victorville, CA)",List(),youtube,1593615063697854,UA000000106530656
macOS,"List(null, null, null)",faq,1593616650732534,1593616950510558,"List(Wichita, KS)",List(),instagram,1593616650732534,UA000000106539566
iOS,"List(null, null, null)",email_coupon,1593618897029648,1593619509779272,"List(San Francisco, CA)",List(),youtube,1593618897029648,UA000000106552410



## 열 표현식

<a href="https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/column.html" target="_blank">열</a>은 표현식을 사용하여 DataFrame의 데이터를 기반으로 계산되는 논리적 구문입니다.

DataFrame의 기존 열을 기반으로 새 열을 생성합니다.

In [0]:
from pyspark.sql.functions import col

print(events_df.device)
print(events_df["device"])
print(col("device"))

Column<'device'>
Column<'device'>
Column<'device'>


Scala는 DataFrame의 기존 열을 기반으로 새 열을 생성하는 추가 구문을 지원합니다.

In [0]:
%scala
$"device"

res0: org.apache.spark.sql.ColumnName = device


### 열 연산자 및 메서드
| 메서드 | 설명 |
| --- | --- |
| \*, + , <, >= | 수학 및 비교 연산자 |
| ==, != | 같음 및 같지 않음 테스트(Scala 연산자는 **`===`** 및 **`=!=`**입니다) |
| alias | 열에 별칭을 지정합니다 |
| cast, astype | 열을 다른 데이터 유형으로 변환합니다 |
| isNull, isNotNull, isNan | null인지, null이 아닌지, NaN인지 |
| asc, desc | 열의 오름차순/내림차순 정렬 표현식을 반환합니다 |


columns, operators, and methods를 사용하여 복잡한 표현식을 만듭니다.

In [0]:
col("ecommerce.purchase_revenue_in_usd") + col("ecommerce.total_item_quantity")
col("event_timestamp").desc()
(col("ecommerce.purchase_revenue_in_usd") * 100).cast("int")

Out[12]: Column<'CAST((ecommerce.purchase_revenue_in_usd * 100) AS INT)'>

다음은 DataFrame 컨텍스트에서 이러한 열 표현식을 사용하는 예입니다.

In [0]:
rev_df = (events_df
         .filter(col("ecommerce.purchase_revenue_in_usd").isNotNull())
         .withColumn("purchase_revenue", (col("ecommerce.purchase_revenue_in_usd") * 100).cast("int"))
         .withColumn("avg_purchase_revenue", col("ecommerce.purchase_revenue_in_usd") / col("ecommerce.total_item_quantity"))
         .sort(col("avg_purchase_revenue").desc())
        )

display(rev_df.limit(10))

device,ecommerce,event_name,event_previous_timestamp,event_timestamp,geo,items,traffic_source,user_first_touch_timestamp,user_id,purchase_revenue,avg_purchase_revenue
Android,"List(1995.0, 1, 1)",finalize,1593302456252258,1593303129418067,"List(Huntsville, AL)","List(List(null, M_PREM_K, Premium King Mattress, 1995.0, 1995.0, 1))",facebook,1593300109279198,UA000000105695674,199500,1995.0
Windows,"List(1995.0, 1, 1)",finalize,1593791895303186,1593792287674884,"List(Enterprise, AL)","List(List(null, M_PREM_K, Premium King Mattress, 1995.0, 1995.0, 1))",google,1593789336454918,UA000000107065404,199500,1995.0
Android,"List(1995.0, 1, 1)",finalize,1593332475475614,1593333526358534,"List(Las Vegas, NV)","List(List(null, M_PREM_K, Premium King Mattress, 1995.0, 1995.0, 1))",instagram,1593330387039056,UA000000105717595,199500,1995.0
iOS,"List(1995.0, 1, 1)",finalize,1592758854477321,1592758945634886,"List(Evergreen, AL)","List(List(null, M_PREM_K, Premium King Mattress, 1995.0, 1995.0, 1))",google,1592755976672897,UA000000104062062,199500,1995.0
Windows,"List(1995.0, 1, 1)",finalize,1593334303652994,1593334388175490,"List(Ann Arbor, MI)","List(List(null, M_PREM_K, Premium King Mattress, 1995.0, 1995.0, 1))",google,1593330407876933,UA000000105717612,199500,1995.0
iOS,"List(1995.0, 1, 1)",finalize,1593712066091989,1593712669023873,"List(Charlotte, NC)","List(List(null, M_PREM_K, Premium King Mattress, 1995.0, 1995.0, 1))",facebook,1593710682055832,UA000000106848770,199500,1995.0
Android,"List(1995.0, 1, 1)",finalize,1593441020743270,1593441200881262,"List(New York, NY)","List(List(null, M_PREM_K, Premium King Mattress, 1995.0, 1995.0, 1))",google,1593439165100482,UA000000106032193,199500,1995.0
Android,"List(1995.0, 1, 1)",finalize,1593806962170483,1593806995657934,"List(Sioux Falls, SD)","List(List(null, M_PREM_K, Premium King Mattress, 1995.0, 1995.0, 1))",google,1593805403702408,UA000000107169375,199500,1995.0
Linux,"List(1995.0, 1, 1)",finalize,1592651332241748,1592651396462843,"List(Lakeville, MN)","List(List(null, M_PREM_K, Premium King Mattress, 1995.0, 1995.0, 1))",google,1592649535031242,UA000000103610853,199500,1995.0
macOS,"List(1995.0, 1, 1)",finalize,1592941603667578,1592941691351054,"List(Grand Junction, CO)","List(List(null, M_PREM_K, Premium King Mattress, 1995.0, 1995.0, 1))",instagram,1592940165877118,UA000000104637395,199500,1995.0



## DataFrame 변환 메서드
| 메서드 | 설명 |
| --- | --- |
| **`select`** | 각 요소에 대해 주어진 표현식을 계산하여 새 DataFrame을 반환합니다. |
| **`drop`** | 열을 삭제한 새 DataFrame을 반환합니다. |
| **`withColumnRenamed`** | 열 이름이 변경된 새 DataFrame을 반환합니다. |
| **`withColumn`** | 열을 추가하거나 이름이 같은 기존 열을 대체하여 새 DataFrame을 반환합니다. |
| **`filter`**, **`where`** | 주어진 조건을 사용하여 행을 필터링합니다. |
| **`sort`**, **`orderBy`** | 주어진 표현식으로 정렬된 새 DataFrame을 반환합니다. |
| **`dropDuplicates`**, **`distinct`** | 중복 행을 제거한 새 DataFrame을 반환합니다. |
| **`limit`** | 처음 n개 행을 가져와 새 DataFrame을 반환합니다. |
| **`groupBy`** | 지정된 열을 사용하여 DataFrame을 그룹화하여 해당 열에 대한 집계를 실행할 수 있습니다. |


### 열 부분 집합
DataFrame 변환을 사용하여 열 부분 집합 만들기


#### **`select()`**
열 목록 또는 열 기반 표현식을 선택합니다.

In [0]:
devices_df = events_df.select("user_id", "device")
display(devices_df.limit(10))

user_id,device
UA000000106525232,Android
UA000000106524533,Android
UA000000106466918,Android
UA000000106513930,Linux
UA000000106522140,Linux
UA000000106541016,macOS
UA000000106541282,macOS
UA000000106530656,iOS
UA000000106539566,macOS
UA000000106552410,iOS


In [0]:
from pyspark.sql.functions import col

locations_df = events_df.select(
    "user_id", 
    col("geo.city").alias("city"), 
    col("geo.state").alias("state")
)
display(locations_df.limit(10))

user_id,city,state
UA000000106525232,Attleboro,MA
UA000000106524533,Rockport,TX
UA000000106466918,De Kalb,TX
UA000000106513930,Warwick,RI
UA000000106522140,Coeur d'Alene,ID
UA000000106541016,Clovis,CA
UA000000106541282,St. Augustine,FL
UA000000106530656,Victorville,CA
UA000000106539566,Wichita,KS
UA000000106552410,San Francisco,CA



#### **`selectExpr()`**
SQL 표현식 목록을 선택합니다.

In [0]:
apple_df = events_df.selectExpr("user_id", "device in ('macOS', 'iOS') as apple_user")
display(apple_df.limit(10))

user_id,apple_user
UA000000106525232,false
UA000000106524533,false
UA000000106466918,false
UA000000106513930,false
UA000000106522140,false
UA000000106541016,true
UA000000106541282,true
UA000000106530656,true
UA000000106539566,true
UA000000106552410,true



#### **`drop()`**
주어진 열을 삭제한 후 새 DataFrame을 반환합니다. 문자열 또는 Column 객체로 지정됩니다.

문자열을 사용하여 여러 열을 지정합니다.

In [0]:
anonymous_df = events_df.drop("user_id", "geo", "device")
display(anonymous_df.limit(10))

ecommerce,event_name,event_previous_timestamp,event_timestamp,items,traffic_source,user_first_touch_timestamp
"List(null, null, null)",mattresses,1593614063954129,1593614089359899,List(),google,1593614037088511
"List(null, null, null)",main,null,1593613901473050,List(),google,1593613901473050
"List(null, null, null)",add_item,1593595595984555,1593595620017592,"List(List(null, M_STAN_Q, Standard Queen Mattress, 1045.0, 1045.0, 1))",instagram,1593595391984879
"List(null, null, null)",main,null,1593611755190083,List(),google,1593611755190083
"List(null, null, null)",mattresses,null,1593613449856735,List(),youtube,1593613449856735
"List(null, null, null)",guest,1593617658665328,1593617822519726,"List(List(null, M_STAN_T, Standard Twin Mattress, 595.0, 595.0, 1))",facebook,1593616922295696
"List(null, null, null)",email_coupon,1593617459515520,1593618432116637,List(),google,1593616968692210
"List(null, null, null)",main,null,1593615063697854,List(),youtube,1593615063697854
"List(null, null, null)",faq,1593616650732534,1593616950510558,List(),instagram,1593616650732534
"List(null, null, null)",email_coupon,1593618897029648,1593619509779272,List(),youtube,1593618897029648


In [0]:
no_sales_df = events_df.drop(col("ecommerce"))
display(no_sales_df.limit(10))

device,event_name,event_previous_timestamp,event_timestamp,geo,items,traffic_source,user_first_touch_timestamp,user_id
Android,mattresses,1593614063954129,1593614089359899,"List(Attleboro, MA)",List(),google,1593614037088511,UA000000106525232
Android,main,null,1593613901473050,"List(Rockport, TX)",List(),google,1593613901473050,UA000000106524533
Android,add_item,1593595595984555,1593595620017592,"List(De Kalb, TX)","List(List(null, M_STAN_Q, Standard Queen Mattress, 1045.0, 1045.0, 1))",instagram,1593595391984879,UA000000106466918
Linux,main,null,1593611755190083,"List(Warwick, RI)",List(),google,1593611755190083,UA000000106513930
Linux,mattresses,null,1593613449856735,"List(Coeur d'Alene, ID)",List(),youtube,1593613449856735,UA000000106522140
macOS,guest,1593617658665328,1593617822519726,"List(Clovis, CA)","List(List(null, M_STAN_T, Standard Twin Mattress, 595.0, 595.0, 1))",facebook,1593616922295696,UA000000106541016
macOS,email_coupon,1593617459515520,1593618432116637,"List(St. Augustine, FL)",List(),google,1593616968692210,UA000000106541282
iOS,main,null,1593615063697854,"List(Victorville, CA)",List(),youtube,1593615063697854,UA000000106530656
macOS,faq,1593616650732534,1593616950510558,"List(Wichita, KS)",List(),instagram,1593616650732534,UA000000106539566
iOS,email_coupon,1593618897029648,1593619509779272,"List(San Francisco, CA)",List(),youtube,1593618897029648,UA000000106552410



### 열 추가 또는 바꾸기
DataFrame 변환을 사용하여 열을 추가하거나 바꿉니다.


#### **`withColumn()`**
같은 이름의 열을 추가하거나 기존 열을 대체하여 새 DataFrame을 반환합니다.

In [0]:
mobile_df = events_df.withColumn("mobile", col("device").isin("iOS", "Android"))
display(mobile_df.limit(10))

device,ecommerce,event_name,event_previous_timestamp,event_timestamp,geo,items,traffic_source,user_first_touch_timestamp,user_id,mobile
Android,"List(null, null, null)",mattresses,1593614063954129,1593614089359899,"List(Attleboro, MA)",List(),google,1593614037088511,UA000000106525232,true
Android,"List(null, null, null)",main,null,1593613901473050,"List(Rockport, TX)",List(),google,1593613901473050,UA000000106524533,true
Android,"List(null, null, null)",add_item,1593595595984555,1593595620017592,"List(De Kalb, TX)","List(List(null, M_STAN_Q, Standard Queen Mattress, 1045.0, 1045.0, 1))",instagram,1593595391984879,UA000000106466918,true
Linux,"List(null, null, null)",main,null,1593611755190083,"List(Warwick, RI)",List(),google,1593611755190083,UA000000106513930,false
Linux,"List(null, null, null)",mattresses,null,1593613449856735,"List(Coeur d'Alene, ID)",List(),youtube,1593613449856735,UA000000106522140,false
macOS,"List(null, null, null)",guest,1593617658665328,1593617822519726,"List(Clovis, CA)","List(List(null, M_STAN_T, Standard Twin Mattress, 595.0, 595.0, 1))",facebook,1593616922295696,UA000000106541016,false
macOS,"List(null, null, null)",email_coupon,1593617459515520,1593618432116637,"List(St. Augustine, FL)",List(),google,1593616968692210,UA000000106541282,false
iOS,"List(null, null, null)",main,null,1593615063697854,"List(Victorville, CA)",List(),youtube,1593615063697854,UA000000106530656,true
macOS,"List(null, null, null)",faq,1593616650732534,1593616950510558,"List(Wichita, KS)",List(),instagram,1593616650732534,UA000000106539566,false
iOS,"List(null, null, null)",email_coupon,1593618897029648,1593619509779272,"List(San Francisco, CA)",List(),youtube,1593618897029648,UA000000106552410,true


In [0]:
purchase_quantity_df = events_df.withColumn("purchase_quantity", col("ecommerce.total_item_quantity").cast("int"))
purchase_quantity_df.printSchema()

root
 |-- device: string (nullable = true)
 |-- ecommerce: struct (nullable = true)
 |    |-- purchase_revenue_in_usd: double (nullable = true)
 |    |-- total_item_quantity: long (nullable = true)
 |    |-- unique_items: long (nullable = true)
 |-- event_name: string (nullable = true)
 |-- event_previous_timestamp: long (nullable = true)
 |-- event_timestamp: long (nullable = true)
 |-- geo: struct (nullable = true)
 |    |-- city: string (nullable = true)
 |    |-- state: string (nullable = true)
 |-- items: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- coupon: string (nullable = true)
 |    |    |-- item_id: string (nullable = true)
 |    |    |-- item_name: string (nullable = true)
 |    |    |-- item_revenue_in_usd: double (nullable = true)
 |    |    |-- price_in_usd: double (nullable = true)
 |    |    |-- quantity: long (nullable = true)
 |-- traffic_source: string (nullable = true)
 |-- user_first_touch_timestamp: long (nullable = true)


#### **`withColumnRenamed()`**
열 이름이 변경된 새 DataFrame을 반환합니다.

In [0]:
location_df = events_df.withColumnRenamed("geo", "location")
display(location_df.limit(10))

device,ecommerce,event_name,event_previous_timestamp,event_timestamp,location,items,traffic_source,user_first_touch_timestamp,user_id
Android,"List(null, null, null)",mattresses,1593614063954129,1593614089359899,"List(Attleboro, MA)",List(),google,1593614037088511,UA000000106525232
Android,"List(null, null, null)",main,null,1593613901473050,"List(Rockport, TX)",List(),google,1593613901473050,UA000000106524533
Android,"List(null, null, null)",add_item,1593595595984555,1593595620017592,"List(De Kalb, TX)","List(List(null, M_STAN_Q, Standard Queen Mattress, 1045.0, 1045.0, 1))",instagram,1593595391984879,UA000000106466918
Linux,"List(null, null, null)",main,null,1593611755190083,"List(Warwick, RI)",List(),google,1593611755190083,UA000000106513930
Linux,"List(null, null, null)",mattresses,null,1593613449856735,"List(Coeur d'Alene, ID)",List(),youtube,1593613449856735,UA000000106522140
macOS,"List(null, null, null)",guest,1593617658665328,1593617822519726,"List(Clovis, CA)","List(List(null, M_STAN_T, Standard Twin Mattress, 595.0, 595.0, 1))",facebook,1593616922295696,UA000000106541016
macOS,"List(null, null, null)",email_coupon,1593617459515520,1593618432116637,"List(St. Augustine, FL)",List(),google,1593616968692210,UA000000106541282
iOS,"List(null, null, null)",main,null,1593615063697854,"List(Victorville, CA)",List(),youtube,1593615063697854,UA000000106530656
macOS,"List(null, null, null)",faq,1593616650732534,1593616950510558,"List(Wichita, KS)",List(),instagram,1593616650732534,UA000000106539566
iOS,"List(null, null, null)",email_coupon,1593618897029648,1593619509779272,"List(San Francisco, CA)",List(),youtube,1593618897029648,UA000000106552410



### 행 부분 집합
DataFrame 변환을 사용하여 행 부분 집합을 만듭니다.


#### **`filter()`**
주어진 SQL 표현식 또는 열 기반 조건을 사용하여 행을 필터링합니다.

##### 별칭: **`where`**

In [0]:
purchases_df = events_df.filter("ecommerce.total_item_quantity > 0")
display(purchases_df.limit(10))

device,ecommerce,event_name,event_previous_timestamp,event_timestamp,geo,items,traffic_source,user_first_touch_timestamp,user_id
Windows,"List(1795.0, 1, 1)",finalize,1593622944339060,1593623248104571,"List(Corpus Christi, TX)","List(List(null, M_PREM_Q, Premium Queen Mattress, 1795.0, 1795.0, 1))",facebook,1593616330759799,UA000000106537762
macOS,"List(1045.0, 1, 1)",finalize,1593614860452264,1593615004570770,"List(Kearney, NE)","List(List(null, M_STAN_Q, Standard Queen Mattress, 1045.0, 1045.0, 1))",instagram,1593610742063884,UA000000106509088
Windows,"List(535.5, 1, 1)",finalize,1593816370279527,1593816433585044,"List(Phoenix, AZ)","List(List(NEWBED10, M_STAN_T, Standard Twin Mattress, 535.5, 595.0, 1))",email,1593608758511600,UA000000106500611
Android,"List(1095.0, 1, 1)",finalize,1593620776166531,1593621612951939,"List(Albuquerque, NM)","List(List(null, M_PREM_T, Premium Twin Mattress, 1095.0, 1095.0, 1))",google,1593616245128899,UA000000106537277
Windows,"List(940.5, 1, 1)",finalize,1593806700141558,1593807430173589,"List(Dallas, TX)","List(List(NEWBED10, M_STAN_Q, Standard Queen Mattress, 940.5, 1045.0, 1))",email,1593606856891824,UA000000106493435
macOS,"List(1995.0, 1, 1)",finalize,1593588128101005,1593588169734017,"List(Clearwater, FL)","List(List(null, M_PREM_K, Premium King Mattress, 1995.0, 1995.0, 1))",email,1593585605089308,UA000000106460128
macOS,"List(940.5, 1, 1)",finalize,1593685522565145,1593686110618872,"List(Kansas City, MO)","List(List(NEWBED10, M_STAN_Q, Standard Queen Mattress, 940.5, 1045.0, 1))",email,1593603980046403,UA000000106484209
macOS,"List(1045.0, 1, 1)",finalize,1593602825823702,1593607138856941,"List(Breckenridge, TX)","List(List(null, M_STAN_Q, Standard Queen Mattress, 1045.0, 1045.0, 1))",facebook,1593598423581032,UA000000106471468
iOS,"List(1695.0, 1, 1)",finalize,1593616170661327,1593616175423652,"List(Newark, NJ)","List(List(null, M_PREM_F, Premium Full Mattress, 1695.0, 1695.0, 1))",instagram,1593613743747321,UA000000106523740
iOS,"List(1045.0, 1, 1)",finalize,1593612481929710,1593612973155915,"List(Raleigh, NC)","List(List(null, M_STAN_Q, Standard Queen Mattress, 1045.0, 1045.0, 1))",facebook,1593610433648747,UA000000106507739


In [0]:
revenue_df = events_df.filter(col("ecommerce.purchase_revenue_in_usd").isNotNull())
display(revenue_df.limit(10))

device,ecommerce,event_name,event_previous_timestamp,event_timestamp,geo,items,traffic_source,user_first_touch_timestamp,user_id
Windows,"List(1795.0, 1, 1)",finalize,1593622944339060,1593623248104571,"List(Corpus Christi, TX)","List(List(null, M_PREM_Q, Premium Queen Mattress, 1795.0, 1795.0, 1))",facebook,1593616330759799,UA000000106537762
macOS,"List(1045.0, 1, 1)",finalize,1593614860452264,1593615004570770,"List(Kearney, NE)","List(List(null, M_STAN_Q, Standard Queen Mattress, 1045.0, 1045.0, 1))",instagram,1593610742063884,UA000000106509088
Windows,"List(535.5, 1, 1)",finalize,1593816370279527,1593816433585044,"List(Phoenix, AZ)","List(List(NEWBED10, M_STAN_T, Standard Twin Mattress, 535.5, 595.0, 1))",email,1593608758511600,UA000000106500611
Android,"List(1095.0, 1, 1)",finalize,1593620776166531,1593621612951939,"List(Albuquerque, NM)","List(List(null, M_PREM_T, Premium Twin Mattress, 1095.0, 1095.0, 1))",google,1593616245128899,UA000000106537277
Windows,"List(940.5, 1, 1)",finalize,1593806700141558,1593807430173589,"List(Dallas, TX)","List(List(NEWBED10, M_STAN_Q, Standard Queen Mattress, 940.5, 1045.0, 1))",email,1593606856891824,UA000000106493435
macOS,"List(1995.0, 1, 1)",finalize,1593588128101005,1593588169734017,"List(Clearwater, FL)","List(List(null, M_PREM_K, Premium King Mattress, 1995.0, 1995.0, 1))",email,1593585605089308,UA000000106460128
macOS,"List(940.5, 1, 1)",finalize,1593685522565145,1593686110618872,"List(Kansas City, MO)","List(List(NEWBED10, M_STAN_Q, Standard Queen Mattress, 940.5, 1045.0, 1))",email,1593603980046403,UA000000106484209
macOS,"List(1045.0, 1, 1)",finalize,1593602825823702,1593607138856941,"List(Breckenridge, TX)","List(List(null, M_STAN_Q, Standard Queen Mattress, 1045.0, 1045.0, 1))",facebook,1593598423581032,UA000000106471468
iOS,"List(1695.0, 1, 1)",finalize,1593616170661327,1593616175423652,"List(Newark, NJ)","List(List(null, M_PREM_F, Premium Full Mattress, 1695.0, 1695.0, 1))",instagram,1593613743747321,UA000000106523740
iOS,"List(1045.0, 1, 1)",finalize,1593612481929710,1593612973155915,"List(Raleigh, NC)","List(List(null, M_STAN_Q, Standard Queen Mattress, 1045.0, 1045.0, 1))",facebook,1593610433648747,UA000000106507739


In [0]:
android_df = events_df.filter((col("traffic_source") != "direct") & (col("device") == "Android"))
display(android_df.limit(10))

device,ecommerce,event_name,event_previous_timestamp,event_timestamp,geo,items,traffic_source,user_first_touch_timestamp,user_id
Android,"List(null, null, null)",mattresses,1593614063954129,1593614089359899,"List(Attleboro, MA)",List(),google,1593614037088511,UA000000106525232
Android,"List(null, null, null)",main,null,1593613901473050,"List(Rockport, TX)",List(),google,1593613901473050,UA000000106524533
Android,"List(null, null, null)",add_item,1593595595984555,1593595620017592,"List(De Kalb, TX)","List(List(null, M_STAN_Q, Standard Queen Mattress, 1045.0, 1045.0, 1))",instagram,1593595391984879,UA000000106466918
Android,"List(null, null, null)",guest,1593618239867244,1593619763160248,"List(Lynn, MA)","List(List(null, M_STAN_Q, Standard Queen Mattress, 1045.0, 1045.0, 1))",facebook,1593617573293065,UA000000106544798
Android,"List(null, null, null)",main,null,1593614903067165,"List(Coos Bay, OR)",List(),google,1593614903067165,UA000000106529769
Android,"List(null, null, null)",main,null,1593577461949983,"List(Knoxville, IA)",List(),instagram,1593577461949983,UA000000106458103
Android,"List(null, null, null)",mattresses,1593608961135558,1593610162266602,"List(New York, NY)",List(),instagram,1593608961135558,UA000000106501454
Android,"List(null, null, null)",main,null,1593614833841187,"List(Long Beach, CA)",List(),google,1593614833841187,UA000000106529402
Android,"List(null, null, null)",reviews,1593601276022194,1593601308453027,"List(Oklahoma City, OK)",List(),facebook,1593601276022194,UA000000106477129
Android,"List(null, null, null)",main,null,1593604330218192,"List(East St. Louis, IL)",List(),google,1593604330218192,UA000000106485219



#### **`dropDuplicates()`**
중복 행을 제거한 새 DataFrame을 반환합니다. 선택적으로 일부 열만 고려합니다.

##### 별칭: **`distinct`**

In [0]:
display(events_df.distinct().limit(10))

device,ecommerce,event_name,event_previous_timestamp,event_timestamp,geo,items,traffic_source,user_first_touch_timestamp,user_id
macOS,"List(null, null, null)",delivery,1593615961875507,1593617767125033,"List(Los Angeles, CA)",List(),google,1593615315744977,UA000000106532025
Windows,"List(null, null, null)",checkout,1593609238501640,1593609449731105,"List(Roseville, CA)","List(List(null, M_STAN_T, Standard Twin Mattress, 595.0, 595.0, 1))",google,1593608512614271,UA000000106499591
macOS,"List(null, null, null)",guest,1593620273632075,1593621142800049,"List(Santa Ana, CA)","List(List(null, M_STAN_K, Standard King Mattress, 1195.0, 1195.0, 1))",direct,1593619294081105,UA000000106554766
Android,"List(null, null, null)",mattresses,null,1593599482228533,"List(Plano, TX)",List(),google,1593599482228533,UA000000106473340
Windows,"List(null, null, null)",main,null,1593614208224712,"List(Shreveport, LA)",List(),youtube,1593614208224712,UA000000106526137
Android,"List(null, null, null)",main,null,1593168828842832,"List(Callaway, FL)",List(),google,1593168828842832,UA000000105196585
iOS,"List(null, null, null)",warranty,1593178048180422,1593178201617547,"List(South San Francisco, CA)",List(),facebook,1593178048180422,UA000000105226624
macOS,"List(null, null, null)",email_coupon,1593176772419899,1593177159499442,"List(Hillsboro, OH)",List(),youtube,1593176772419899,UA000000105221278
Windows,"List(null, null, null)",mattresses,null,1593180840010479,"List(Yuma, AZ)",List(),youtube,1593180840010479,UA000000105239807
iOS,"List(null, null, null)",add_item,1593182615377342,1593184716244084,"List(Lampasas, TX)","List(List(null, M_STAN_T, Standard Twin Mattress, 595.0, 595.0, 1))",google,1593182426293752,UA000000105248118


In [0]:
distinct_users_df = events_df.dropDuplicates(["user_id"])
display(distinct_users_df.limit(10))

device,ecommerce,event_name,event_previous_timestamp,event_timestamp,geo,items,traffic_source,user_first_touch_timestamp,user_id
iOS,"List(2646.0, 2, 2)",finalize,1592558744336377,1592558997140898,"List(New Bern, NC)","List(List(NEWBED10, M_PREM_K, Premium King Mattress, 1795.5, 1995.0, 1), List(NEWBED10, M_STAN_F, Standard Full Mattress, 850.5, 945.0, 1))",email,1592214370993088,UA000000102368951
macOS,"List(null, null, null)",checkout,1592627411180086,1592627415119183,"List(Cookeville, TN)","List(List(NEWBED10, M_STAN_F, Standard Full Mattress, 850.5, 945.0, 1))",email,1592218408760826,UA000000102377152
macOS,"List(957.6, 2, 2)",finalize,1592538154825817,1592540091048118,"List(New York, NY)","List(List(NEWBED10, M_STAN_F, Standard Full Mattress, 850.5, 945.0, 1), List(NEWBED10, P_DOWN_S, Standard Down Pillow, 107.10000000000001, 119.0, 1))",email,1592222167471834,UA000000102388362
Android,"List(null, null, null)",mattresses,1592223503585690,1592571872354183,"List(Hodgenville, KY)",List(),email,1592223451843181,UA000000102392998
macOS,"List(535.5, 1, 1)",finalize,1592545976095877,1592546695363151,"List(Inkster, MI)","List(List(NEWBED10, M_STAN_T, Standard Twin Mattress, 535.5, 595.0, 1))",email,1592232212733578,UA000000102435405
Android,"List(null, null, null)",shipping_info,1592596272117609,1592596840081876,"List(New York, NY)","List(List(NEWBED10, M_STAN_T, Standard Twin Mattress, 535.5, 595.0, 1), List(NEWBED10, P_FOAM_S, Standard Foam Pillow, 53.1, 59.0, 1), List(NEWBED10, M_STAN_Q, Standard Queen Mattress, 940.5, 1045.0, 1))",email,1592236655400157,UA000000102462564
Android,"List(null, null, null)",cc_info,1592683907107987,1592684037498241,"List(Lafayette, TN)","List(List(NEWBED10, M_PREM_T, Premium Twin Mattress, 985.5, 1095.0, 1), List(NEWBED10, P_FOAM_S, Standard Foam Pillow, 53.1, 59.0, 1))",email,1592236949481874,UA000000102464478
Android,"List(null, null, null)",checkout,1592546356440395,1592546449770388,"List(Santa Ana, CA)","List(List(NEWBED10, M_STAN_Q, Standard Queen Mattress, 940.5, 1045.0, 1))",email,1592238994903828,UA000000102477568
Linux,"List(null, null, null)",register,1592539759933166,1592539762466183,"List(Raleigh, NC)","List(List(NEWBED10, M_STAN_Q, Standard Queen Mattress, 940.5, 1045.0, 1))",email,1592245004895189,UA000000102515405
Android,"List(null, null, null)",cart,1592601007755261,1592601547959528,"List(Kansas City, MO)","List(List(NEWBED10, M_PREM_F, Premium Full Mattress, 1525.5, 1695.0, 1))",email,1592245272454879,UA000000102516945



#### **`limit()`**
처음 n개 행을 가져와 새 DataFrame을 반환합니다.

In [0]:
limit_df = events_df.limit(100)
display(limit_df.limit(10))

device,ecommerce,event_name,event_previous_timestamp,event_timestamp,geo,items,traffic_source,user_first_touch_timestamp,user_id
Android,"List(null, null, null)",mattresses,1593614063954129,1593614089359899,"List(Attleboro, MA)",List(),google,1593614037088511,UA000000106525232
Android,"List(null, null, null)",main,null,1593613901473050,"List(Rockport, TX)",List(),google,1593613901473050,UA000000106524533
Android,"List(null, null, null)",add_item,1593595595984555,1593595620017592,"List(De Kalb, TX)","List(List(null, M_STAN_Q, Standard Queen Mattress, 1045.0, 1045.0, 1))",instagram,1593595391984879,UA000000106466918
Linux,"List(null, null, null)",main,null,1593611755190083,"List(Warwick, RI)",List(),google,1593611755190083,UA000000106513930
Linux,"List(null, null, null)",mattresses,null,1593613449856735,"List(Coeur d'Alene, ID)",List(),youtube,1593613449856735,UA000000106522140
macOS,"List(null, null, null)",guest,1593617658665328,1593617822519726,"List(Clovis, CA)","List(List(null, M_STAN_T, Standard Twin Mattress, 595.0, 595.0, 1))",facebook,1593616922295696,UA000000106541016
macOS,"List(null, null, null)",email_coupon,1593617459515520,1593618432116637,"List(St. Augustine, FL)",List(),google,1593616968692210,UA000000106541282
iOS,"List(null, null, null)",main,null,1593615063697854,"List(Victorville, CA)",List(),youtube,1593615063697854,UA000000106530656
macOS,"List(null, null, null)",faq,1593616650732534,1593616950510558,"List(Wichita, KS)",List(),instagram,1593616650732534,UA000000106539566
iOS,"List(null, null, null)",email_coupon,1593618897029648,1593619509779272,"List(San Francisco, CA)",List(),youtube,1593618897029648,UA000000106552410



### 행 정렬
DataFrame 변환을 사용하여 행을 정렬합니다.


#### **`sort()`**
주어진 열 또는 표현식을 기준으로 정렬된 새 DataFrame을 반환합니다.

##### 별칭: **`orderBy`**

In [0]:
increase_timestamps_df = events_df.sort("event_timestamp")
display(increase_timestamps_df.limit(10))

device,ecommerce,event_name,event_previous_timestamp,event_timestamp,geo,items,traffic_source,user_first_touch_timestamp,user_id
Android,"List(null, null, null)",cart,1592539157333154,1592539212858607,"List(Miami Beach, FL)","List(List(NEWBED10, M_STAN_T, Standard Twin Mattress, 535.5, 595.0, 1))",email,1592294681843997,UA000000102604475
Windows,"List(null, null, null)",checkout,1592538308929545,1592539217303275,"List(New Orleans, LA)","List(List(NEWBED10, M_STAN_F, Standard Full Mattress, 850.5, 945.0, 1))",email,1592317381269186,UA000000102665598
macOS,"List(null, null, null)",mattresses,1592232855039331,1592539236295422,"List(Inkster, MI)",List(),email,1592232212733578,UA000000102435405
macOS,"List(null, null, null)",add_item,1592539236295422,1592539310482088,"List(Inkster, MI)","List(List(NEWBED10, M_STAN_T, Standard Twin Mattress, 535.5, 595.0, 1))",email,1592232212733578,UA000000102435405
Chrome OS,"List(1195.0, 1, 1)",finalize,1592537456611886,1592539318119643,"List(Scottsbluff, NE)","List(List(null, M_STAN_K, Standard King Mattress, 1195.0, 1195.0, 1))",google,1592533311272204,UA000000103312488
Windows,"List(null, null, null)",main,null,1592539320276389,"List(San Francisco, CA)",List(),google,1592539320276389,UA000000103314669
Windows,"List(null, null, null)",faq,1592539320276389,1592539322539918,"List(San Francisco, CA)",List(),google,1592539320276389,UA000000103314669
Android,"List(null, null, null)",main,null,1592539360387580,"List(Henderson, NV)",List(),google,1592539360387580,UA000000103314680
Windows,"List(null, null, null)",add_item,1592538356374145,1592539361443616,"List(Virginia Beach, VA)","List(List(NEWBED10, M_STAN_Q, Standard Queen Mattress, 940.5, 1045.0, 1))",email,1592349890694428,UA000000102833786
Chrome OS,"List(null, null, null)",premium,1592539069931863,1592539395432411,"List(Winthrop Town, MA)",List(),youtube,1592539069931863,UA000000103314621


In [0]:
decrease_timestamp_df = events_df.sort(col("event_timestamp").desc())
display(decrease_timestamp_df.limit(10))

device,ecommerce,event_name,event_previous_timestamp,event_timestamp,geo,items,traffic_source,user_first_touch_timestamp,user_id
Android,"List(null, null, null)",delivery,1593879219241045,1593879299254455,"List(College Station, TX)",List(),instagram,1593879219241045,UA000000107382465
Android,"List(null, null, null)",mattresses,null,1593879298321389,"List(Bakersfield, CA)",List(),google,1593879298321389,UA000000107383208
iOS,"List(null, null, null)",premium,1593879185676895,1593879297580081,"List(Sault Ste. Marie, MI)",List(),facebook,1593878892259305,UA000000107379435
Chrome OS,"List(null, null, null)",mattresses,null,1593879297072235,"List(Statesboro, GA)",List(),google,1593879297072235,UA000000107383196
Linux,"List(null, null, null)",reviews,1593877177459633,1593879296532428,"List(Hialeah, FL)",List(),facebook,1593877177459633,UA000000107363919
Android,"List(null, null, null)",original,1593879228125080,1593879296190547,"List(Albuquerque, NM)",List(),facebook,1593879228125080,UA000000107382532
Linux,"List(null, null, null)",main,null,1593879295204749,"List(Whitewater, WI)",List(),google,1593879295204749,UA000000107383180
Chrome OS,"List(null, null, null)",shipping_info,1593879287705053,1593879289866628,"List(Houston, TX)","List(List(null, M_STAN_Q, Standard Queen Mattress, 1045.0, 1045.0, 1))",google,1593877952339532,UA000000107370981
Windows,"List(null, null, null)",main,null,1593879289049444,"List(Fullerton, CA)",List(),google,1593879289049444,UA000000107383130
Linux,"List(null, null, null)",premium,1593878368246358,1593879288113451,"List(Dyersburg, TN)",List(),direct,1593878368246358,UA000000107374799


In [0]:
increase_sessions_df = events_df.orderBy(["user_first_touch_timestamp", "event_timestamp"])
display(increase_sessions_df.limit(10))

device,ecommerce,event_name,event_previous_timestamp,event_timestamp,geo,items,traffic_source,user_first_touch_timestamp,user_id
iOS,"List(null, null, null)",mattresses,1592205687627986,1592583618219827,"List(Gibraltar, MI)",List(),email,1592205125802184,UA000000102359929
iOS,"List(null, null, null)",add_item,1592583618219827,1592584060201694,"List(Gibraltar, MI)","List(List(NEWBED10, M_STAN_T, Standard Twin Mattress, 535.5, 595.0, 1))",email,1592205125802184,UA000000102359929
iOS,"List(null, null, null)",add_item,1592584060201694,1592584530360118,"List(Gibraltar, MI)","List(List(NEWBED10, M_STAN_T, Standard Twin Mattress, 535.5, 595.0, 1), List(NEWBED10, P_DOWN_S, Standard Down Pillow, 107.10000000000001, 119.0, 1))",email,1592205125802184,UA000000102359929
iOS,"List(null, null, null)",cart,1592584530360118,1592584740438526,"List(Gibraltar, MI)","List(List(NEWBED10, M_STAN_T, Standard Twin Mattress, 535.5, 595.0, 1), List(NEWBED10, P_DOWN_S, Standard Down Pillow, 107.10000000000001, 119.0, 1))",email,1592205125802184,UA000000102359929
iOS,"List(null, null, null)",checkout,1592584740438526,1592584982255874,"List(Gibraltar, MI)","List(List(NEWBED10, M_STAN_T, Standard Twin Mattress, 535.5, 595.0, 1), List(NEWBED10, P_DOWN_S, Standard Down Pillow, 107.10000000000001, 119.0, 1))",email,1592205125802184,UA000000102359929
iOS,"List(null, null, null)",register,1592584982255874,1592586241748725,"List(Gibraltar, MI)","List(List(NEWBED10, M_STAN_T, Standard Twin Mattress, 535.5, 595.0, 1), List(NEWBED10, P_DOWN_S, Standard Down Pillow, 107.10000000000001, 119.0, 1))",email,1592205125802184,UA000000102359929
iOS,"List(null, null, null)",shipping_info,1592586241748725,1592586309662533,"List(Gibraltar, MI)","List(List(NEWBED10, M_STAN_T, Standard Twin Mattress, 535.5, 595.0, 1), List(NEWBED10, P_DOWN_S, Standard Down Pillow, 107.10000000000001, 119.0, 1))",email,1592205125802184,UA000000102359929
iOS,"List(null, null, null)",cc_info,1592586309662533,1592586426868910,"List(Gibraltar, MI)","List(List(NEWBED10, M_STAN_T, Standard Twin Mattress, 535.5, 595.0, 1), List(NEWBED10, P_DOWN_S, Standard Down Pillow, 107.10000000000001, 119.0, 1))",email,1592205125802184,UA000000102359929
iOS,"List(642.6, 2, 2)",finalize,1592586426868910,1592586469839318,"List(Gibraltar, MI)","List(List(NEWBED10, M_STAN_T, Standard Twin Mattress, 535.5, 595.0, 1), List(NEWBED10, P_DOWN_S, Standard Down Pillow, 107.10000000000001, 119.0, 1))",email,1592205125802184,UA000000102359929
Windows,"List(null, null, null)",mattresses,1592216509596763,1592568403437689,"List(New York, NY)",List(),email,1592214239380036,UA000000102368745


In [0]:
decrease_sessions_df = events_df.sort(col("user_first_touch_timestamp").desc(), col("event_timestamp"))
display(decrease_sessions_df.limit(10))

device,ecommerce,event_name,event_previous_timestamp,event_timestamp,geo,items,traffic_source,user_first_touch_timestamp,user_id
Android,"List(null, null, null)",mattresses,null,1593879298321389,"List(Bakersfield, CA)",List(),google,1593879298321389,UA000000107383208
Chrome OS,"List(null, null, null)",mattresses,null,1593879297072235,"List(Statesboro, GA)",List(),google,1593879297072235,UA000000107383196
Linux,"List(null, null, null)",main,null,1593879295204749,"List(Whitewater, WI)",List(),google,1593879295204749,UA000000107383180
Windows,"List(null, null, null)",main,null,1593879289049444,"List(Fullerton, CA)",List(),google,1593879289049444,UA000000107383130
iOS,"List(null, null, null)",main,null,1593879287097367,"List(Laredo, TX)",List(),google,1593879287097367,UA000000107383116
Linux,"List(null, null, null)",mattresses,null,1593879280264292,"List(San Jose, CA)",List(),youtube,1593879280264292,UA000000107383057
Android,"List(null, null, null)",main,null,1593879279163480,"List(East Providence, RI)",List(),google,1593879279163480,UA000000107383039
Windows,"List(null, null, null)",main,null,1593879278766317,"List(Boston, MA)",List(),google,1593879278766317,UA000000107383034
macOS,"List(null, null, null)",pillows,null,1593879278524911,"List(Dallas, TX)",List(),google,1593879278524911,UA000000107383030
iOS,"List(null, null, null)",pillows,null,1593879274597808,"List(Arlington, TX)",List(),google,1593879274597808,UA000000107382992



이 레슨과 관련된 테이블과 파일을 삭제하려면 다음 셀을 실행하세요.

In [0]:
DA.cleanup()

Resetting the learning environment:
| dropping the schema "3dt005_nuk5_da_dewd"...(2 seconds)
| removing the working directory "dbfs:/mnt/dbacademy-users/3dt005@msacademy.msai.kr/data-engineering-with-databricks"...(0 seconds)

Validating the locally installed datasets:
| listing local files...(3 seconds)
| validation completed...(3 seconds total)
